### CArga de bases


In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()



In [2]:
query = f"""
SELECT  *   FROM DANTALION.[dbo].Base_Maestra_Efectiva_Vigente
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [20]:

filename='consumo_ayer.csv'

df_base_1=cargar_archivo_csv(spark,filename,';',True)
filename='consumo_hoy.csv'
df_base_2=cargar_archivo_csv(spark,filename,';',True)




print(df_base_1.columns)
print(df_base_2.columns)


['FECCORTE', 'FECCORTERCC', 'PERIODO', 'DNI', 'CLIENTE', 'ASIGNACION', 'CANAL', 'FECHA', 'ZONA', 'PLAZA', 'CODIGO_AGENCIA', 'AGENCIA', 'DIRECCION', 'URBANIZACION', 'DISTRITO', 'PROVINCIA', 'DEPARTAMENTO', 'TASA', 'LINEA_ACOTADA', 'CME_DISPONIBLE', 'PERFIL', 'SEGMENTO', 'SCORE', 'CODBASE', 'NOMCOMERCIAL', 'FLGREPROGRAMADOCOVID19EFE', 'FECVCTOPROXCUOTACPRC19EFE', 'CELULAR1', 'CELULAR2', 'CELULAR3', 'CELULAR4', 'CELULAR5', 'CELULAR6', 'CELULAR7', 'CELULAR8', 'CELULAR9', 'CELULAR10', 'CELULAR11', 'CELULAR12', 'CELULAR13', 'CELULAR14', 'CELULAR15', 'EMPRESA1', 'EMPRESA2', 'EMPRESA3', 'TIPOCLIENTE', 'CODCANAL', 'FECULTASIGNACION', 'FLGCONVENIO', 'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'FLGSUBPROCESO_HS', 'FLGTIPORETANQUEO', 'LINEA_FT_RETANQUEO', 'LINEA_HS_RS_RETANQUEO', 'LINEA_HS_PLUS_RETANQUEO', 'LINEA_FULL_RETANQUEO', 'SITUACIONLABORAL', 'TIPOINGRESO', 'PERFIL_IC']
['FECCORTE', 'FECCORTERCC', 'PERIODO', 'DNI', 'CLIENTE', 'ASIGNACION', 'CANAL', 'FECHA', 'ZONA', 'PLAZA', 'C

In [21]:
df_base_1 = df_base_1.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
)
df_base_12= df_base_2.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
)

In [22]:
df_base_1=df_base_1.withColumn('FECHA_ENVIO',F.lit('2026-07-07'))
df_base_2=df_base_2.withColumn('FECHA_ENVIO',F.lit('2026-07-08'))

In [23]:
print(df_formato.join(df_base_1,['DNI'],'inner').count())
print(df_formato.join(df_base_2,['DNI'],'inner').count())
print(df_base_1.join(df_base_2,['DNI'],'inner').count())
print(df_base_1.join(df_base_2,['DNI'],'leftanti').count())
print(df_base_2.join(df_base_1,['DNI'],'leftanti').count())

0
0
0
31550
45000


In [24]:
df_base=df_base_1.unionByName(df_base_2)

In [ ]:
exprs = [
    F.count(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c).cast("string")) != "") &
            (F.upper(F.trim(F.col(c).cast("string"))) != "NULL"),
            c
        )
    ).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [26]:
cols_cel = [f"CELULAR{i}" for i in range(1, 15)]

df_base = df_base.withColumn(
    "CELULARES_ARRAY",
    F.array(*[F.col(c).cast("string") for c in cols_cel])
)

df_base = df_base.withColumn(
    "CELULARES_VALIDOS",
    F.array_distinct(
        F.filter(
            F.transform(
                F.col("CELULARES_ARRAY"),
                lambda x: F.regexp_replace(F.trim(x), r"\D", "")
            ),
            lambda x: x.rlike(r"^9\d{8}$")
        )
    )
)

max_cel = 15

for i in range(max_cel):
    df_base = df_base.withColumn(
        f"CEL{str(i+1).zfill(2)}",
        F.expr(f"get(CELULARES_VALIDOS, {i})")
    )

df_base = df_base.drop(
    "CELULARES_ARRAY",
    "CELULARES_VALIDOS",
    *cols_cel
)

In [27]:
df_base=df_base.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base=df_base.withColumn('MES_DURACION_BASE',F.lit('07'))
# df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-07-01'))
df_base=df_base.withColumn('SERVICIO',F.lit('05'))


In [29]:
df_base=df_base.withColumnRenamed('TIPOCLIENTE','tipocliente')
df_base=df_base.withColumnRenamed('EMPRESA1','empresa1')
df_base=df_base.withColumnRenamed('EMPRESA2','empresa2')
df_base=df_base.withColumnRenamed('EMPRESA3','empresa3')

In [31]:
df_base=df_base.drop('FECCORTE', 'FECCORTERCC', 'PERIODO')

In [32]:
cols_base = set(df_base.columns)
cols_formato = set(df_formato.columns)

solo_en_base = cols_base - cols_formato
print("Solo en df_base:", solo_en_base)
solo_en_formato = cols_formato -cols_base
print("Solo en formato:", solo_en_formato)


Solo en df_base: set()
Solo en formato: {'LINEA_HS_PLUS_RETANQUEO', 'TELF6', 'CELULAR7', 'LINEA_FULL', 'TELEFONO', 'CELULAR1', 'CELULAR10', 'FLAT1', 'REP3', 'CELULAR3', 'LINEA_FULL_RETANQUEO', 'TELF7', 'MARCA3', 'TELF2', 'MARCA4', 'CELULAR11', 'CELULAR4', 'MARCA_2025', 'CELULAR5', 'CELULAR14', 'REP4', 'LINEA_FT_RETANQUEO', 'REP2', 'TELF3', 'CELULAR13', 'TELF8', 'TELF5', 'CELULAR8', 'RangoSaldo2', 'RETIRO', 'MARCA2', 'CELULAR9', 'RangoSaldo1', 'RangoSaldo3', 'MARCA5', 'FLAT2', 'TELF1', 'Retail', 'CELULAR12', 'MICROZONA', 'LINEA_HS_RS_RETANQUEO', 'PLAZA', 'CELULAR6', 'TELF4', 'CELULAR2', 'REP1', 'FLAT3', 'MARCA', 'DEVUELTO'}


In [33]:
print(df_base.count())
print(df_base.dropDuplicates(['DNI']).count())

76550
76550


In [34]:
append_table_SQL(spark,df_base,'Base_Maestra_Efectiva',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [35]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva", "SP tNumeros Consumo")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva", "SP actualizar Consumo Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva", "SP actualizar Consumo SA")

SP tNumeros Consumo | realizado | duración: 586.39 seg
SP actualizar Consumo Zeus | realizado | duración: 188.5 seg
SP actualizar Consumo SA | realizado | duración: 14.79 seg
